# Intraday ES Impulse PCA Research Runthrough

This notebook is the working research runthrough for the intraday branch of ChronoSwan. It studies whether large 60-minute ES1 moves have recurring cross-asset drivers, and whether event-conditioned PCA adds anything beyond a simple conditional correlation study.

Raw Bloomberg workbooks and generated outputs are local-only. The public repo should carry the method, code, and unexecuted notebook; executed data-derived reports should stay in `reports/` unless data licensing allows publication.

## Literature Context

This is not a blank-slate idea. The relevant prior work includes extreme-dependence studies, contagion/interdependence measurement, dynamic conditional correlations, spillover networks, and PCA concentration measures of systemic risk.

- Longin and Solnik show that market correlations can behave differently in the tails: https://doi.org/10.1111/0022-1082.00340
- Forbes and Rigobon warn that crisis correlations can be mechanically biased upward by volatility: https://www.nber.org/papers/w7267
- Engle's DCC framework is the formal dynamic-correlation benchmark: https://doi.org/10.1198/073500102288618487
- Diebold and Yilmaz motivate directional spillover analysis as a later extension: https://doi.org/10.1016/j.ijforecast.2012.08.006
- Kritzman, Li, Page, and Rigobon use PCA concentration as a systemic-risk measure: https://doi.org/10.2469/faj.v67.n1.5

The narrower contribution here is a point-in-time ES impulse workflow: define large moves using a shifted rolling threshold, compare pairwise conditional correlations to event-conditioned PCA, and record driver-attribution tables that are useful in a macro/risk conversation.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from chronoswan.experiments.intraday_impulse_pca import run_intraday_impulse_pca

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

input_path = project_root / "data" / "17sheets.xlsx"
if not input_path.exists():
    input_path = project_root / "data" / "raw" / "17sheets.xlsx"

output_dir = project_root / "data" / "processed"
reports_dir = project_root / "reports"
reports_dir.mkdir(exist_ok=True)

result = run_intraday_impulse_pca(input_path=input_path, output_dir=output_dir)

coverage = result["coverage"]
events = result["events"]
event_summary = result["event_summary"]
corr = result["conditional_correlations"]
pca_summary = result["pca_summary"]
pca_loadings = result["pca_loadings"]
rolling_pca = result["rolling_pca"]
predictive = result["predictive_results"]
coefficients = result["predictive_coefficients"]

print(f"Loaded {coverage['ticker'].nunique()} tickers from {input_path}.")
print(f"Return grid: {result['return_panel'].index.min()} to {result['return_panel'].index.max()}.")

## Data Audit

The first check is whether the workbook is usable as a cross-asset intraday panel. Futures and FX trade around the clock; ETFs and some indices have cash-session-like coverage, so exact timestamp overlap is expected to be thinner for PCA than for pairwise correlations.

In [ ]:
coverage.sort_values("rows", ascending=False)

## Event Definition

A large ES impulse is defined by the absolute 60-minute ES1 log return exceeding the shifted rolling 95th percentile of the prior 20-day window. The shift matters: the current bar cannot set its own threshold.

In [ ]:
event_summary.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
plot_frame = events.dropna(subset=["es_return_1h"])
ax.plot(plot_frame.index, plot_frame["es_return_1h"] * 10_000, linewidth=0.8, label="ES1 60m return, bp")
ax.scatter(
    events.index[events["large_down"]],
    events.loc[events["large_down"], "es_return_1h"] * 10_000,
    s=18,
    label="large down",
)
ax.scatter(
    events.index[events["large_up"]],
    events.loc[events["large_up"], "es_return_1h"] * 10_000,
    s=18,
    label="large up",
)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("ES1 hourly impulse labels")
ax.set_ylabel("basis points")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(reports_dir / "intraday_es_impulse_labels.png", dpi=150)
plt.show()

## Conditional Correlation Benchmark

This is the benchmark your boss explicitly asked about: before PCA, ask whether simple correlations on significant ES moves already tell the driver story.

In [ ]:
for sample in ["threshold_ready", "large_abs", "large_down", "large_up"]:
    print(f"\n{sample}")
    display(
        corr.query("sample == @sample")
        .sort_values(["abs_corr_with_es", "n_obs"], ascending=[False, False])
        .head(12)
        .round(4)
    )

## Event-Conditioned PCA

PCA is fitted on standardized driver returns, excluding ES1. Component signs are aligned so positive loadings are assets that move with positive ES returns; negative loadings are assets that tend to move against ES.

In [ ]:
pca_summary.query("status == 'fit'").round(4)

In [ ]:
for sample in ["threshold_ready", "large_abs", "large_down", "large_up"]:
    top = (
        pca_loadings.query("sample == @sample and component == 1")
        .sort_values("abs_loading", ascending=False)
        .head(10)
        [["driver", "loading_aligned_to_es", "abs_loading"]]
        .round(4)
    )
    print(f"\n{sample}: PC1 loadings")
    display(top)

## Rolling PCA Concentration

The absorption-style statistic is the fraction of standardized driver variance explained by the first three principal components. This is a concentration measure, not a directional forecast.

In [ ]:
rolling_pca[["all_bar_absorption", "large_abs_absorption", "large_abs_rows"]].describe().round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(rolling_pca["timestamp"], rolling_pca["all_bar_absorption"], label="all bars")
ax.plot(rolling_pca["timestamp"], rolling_pca["large_abs_absorption"], label="large ES moves")
ax.set_title("Rolling PCA concentration")
ax.set_ylabel("first 3 PCs variance share")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(reports_dir / "intraday_rolling_pca_absorption.png", dpi=150)
plt.show()

## Predictive Signal Screen

This is a deliberately modest chronological screen. It asks whether known-at-bar-close cross-asset features rank next-bar large ES moves better than the train-set base rate. It is not a production forecast or trading model.

In [ ]:
predictive.round(4)

In [ ]:
for target in coefficients["target"].drop_duplicates():
    for model in coefficients.loc[coefficients["target"].eq(target), "model"].drop_duplicates():
        print(f"\n{target} / {model}")
        display(
            coefficients.query("target == @target and model == @model")
            .head(12)
            .round(4)
        )

## Current Interpretation

The core result to inspect is whether large-move correlations and event-conditioned PCA tell the same story. If they do, PCA is mainly a compression/communication layer. If PCA reveals a stable factor with cross-asset composition that is not obvious from pairwise correlations, then the angle becomes more interesting.

The modeling screen should be judged against the train base-rate row. Rare-event ranking improvements are useful only if calibration is controlled; balanced logistic coefficients are diagnostic, not probabilities to quote.

In [ ]:
large_abs_pc1 = pca_summary.query("sample == 'large_abs' and component == 1").iloc[0]
ready_pc1 = pca_summary.query("sample == 'threshold_ready' and component == 1").iloc[0]
absorption = rolling_pca[["all_bar_absorption", "large_abs_absorption"]].describe()
best_large_down_corr = corr.query("sample == 'large_down'").sort_values("abs_corr_with_es", ascending=False).head(5)
best_abs_model = predictive.query("target == 'target_next_large_abs' and model == 'logit_unweighted'").iloc[0]

print("Research readout")
print(f"- Large absolute ES impulses: PC1 explains {large_abs_pc1['explained_variance_ratio']:.1%} of driver variance versus {ready_pc1['explained_variance_ratio']:.1%} on threshold-ready bars.")
print(f"- Large-move PC1 is aligned with ES at abs corr {large_abs_pc1['abs_corr_with_es']:.2f}.")
print(f"- Median rolling absorption: all bars {absorption.loc['50%', 'all_bar_absorption']:.1%}, large ES moves {absorption.loc['50%', 'large_abs_absorption']:.1%}.")
print("- Top large-down pairwise drivers:")
for _, row in best_large_down_corr.iterrows():
    print(f"  {row['driver']}: corr {row['corr_with_es']:.2f} over {int(row['n_obs'])} observations")
print(f"- Next-bar large-absolute logistic screen: ROC AUC {best_abs_model['roc_auc']:.2f}, AP {best_abs_model['average_precision']:.2%}, Brier {best_abs_model['brier_score']:.4f}.")